# 记忆管理 (Memory Management)

- 有效的记忆管理对于智能体保留信息至关重要。智能体需要不同类型的记忆，就像人类一样，以高效操作。
- 本章深入探讨记忆管理，特别关注智能体的短期（即时）和长期（持久）记忆需求。

- 短期记忆（上下文记忆）：
  - 类似于工作记忆，保存当前处理或最近访问的信息。
  - 对于使用大型语言模型（LLM）的智能体，短期记忆主要存在于上下文窗口中。
  - 上下文窗口包含最近的消息、智能体回复、工具使用结果和来自当前交互的反思，这些内容共同通知LLM的后续响应和行为。
  - 上下文窗口容量有限，限制了智能体可以直接访问的近期信息量。
  - 高效的短期记忆管理可能通过总结对话片段或强调关键细节实现。
  - 通过扩展“长上下文”窗口，模型能够在单次交互中保存更多信息。
  - 但上下文是短暂的，交互结束后即丢失，处理代价较高且效率较低。
  - 因此，智能体需要不同的记忆类型以实现持久性、从过去交互中回忆信息并建立知识库。

- 长期记忆（持久记忆）：
  - 充当智能体在多个交互、任务或延长期限中保留信息的仓库，类似于长期知识库。
  - 数据通常存储在智能体即时处理环境之外，如数据库、知识图谱或向量数据库中。
  - 在向量数据库中，信息以数值向量形式存储，便于智能体基于语义相似度（而非关键词匹配）检索数据，即语义搜索。
  - 当智能体需要长期记忆相关信息时，会查询外部存储，获取相关数据并整合到短期上下文以便即时使用，从而将之前的知识与当前交互结合起来。

## 实际应用与用例

- **聊天机器人和对话式人工智能**：维持对话流依赖短期记忆。聊天机器人需要记住先前的用户输入以提供连贯的响应。长期记忆使聊天机器人能够回忆用户偏好、过去的问题或讨论，提供个性化和持续的互动。
- **任务导向型代理**：管理多步骤任务的代理需要短期记忆来跟踪先前步骤、当前进展和整体目标。此信息可能存在于任务的上下文或临时存储中。长期记忆对于访问不在即时上下文中的特定用户相关数据至关重要。
- **个性化体验**：提供定制化交互的代理利用长期记忆存储和检索用户偏好、行为历史和个人信息，从而使代理能够调整其响应和建议。
- **学习与改进**：代理通过学习过去的互动来提升性能。成功的策略、错误和新信息被存储在长期记忆中，促进未来的适应。强化学习代理以此方式存储学习到的策略或知识。
- **信息检索（RAG）**：设计用于回答问题的代理访问知识库，其长期记忆通常实现为检索增强生成（RAG）。代理检索相关文档或数据以支持其回答。
- **自主系统**：机器人或自动驾驶汽车需要记忆地图、路线、物体位置和学习到的行为。涉及短期记忆以感知周围环境和长期记忆以掌握一般环境知识。

记忆使代理能够维护历史、学习、个性化交互以及管理复杂的时间依赖问题。

## Hands-On 代码：Google Agent Developer Kit (ADK) 中的记忆管理

- Google Agent Developer Kit (ADK) 提供了一种结构化方法来管理上下文和记忆，包括面向实际应用程序的实用方法。对 ADK 的会话、状态和记忆的牢固理解对于构建需要保留信息的代理至关重要。
- 类似于人类的交互，代理能够回忆以前的交流，以进行连贯和自然的对话。ADK 通过三个核心概念及其关联服务简化了上下文管理。
- 每个与代理的交互都可以被视为一个唯一的对话线程。代理可能需要访问早期交互的信息。ADK 将其结构化如下：
  - **Session（会话）**：单个聊天线程，记录消息和动作（事件）的日志，存储与该特定交互相关的临时数据（状态）。
  - **State（状态，session.state）**：存储于会话内，包含与当前激活的聊天线程相关的信息。
  - **Memory（记忆）**：可搜索的信息存储库，来源于各种过去的聊天或外部资源，作为超越即时对话的数据检索资源。

- ADK 提供了管理构建复杂、状态丰富和上下文感知代理所需的关键组件服务：
  - **SessionService 会话服务**：通过处理会话对象的启动、记录和终止来管理聊天线程。
  - **MemoryService 记忆服务**：负责存储和检索长期知识（记忆）。

- 会话服务和记忆服务均支持各种配置选项：
  - 允许使用基于应用程序需求的不同存储方法。
  - 记忆选项用于测试目的，数据不会在重启后持续存在。
  - 持久存储和可扩展性方面，ADK 还支持数据库和云端服务。



## 会话：跟踪每个聊天

- ADK 中的 Session 对象设计用于跟踪和管理单个聊天线程。
- 与代理对话开始时，SessionService 生成一个 Session 对象，表示该对话：
  - 包含特定会话的唯一识别符（例如 app_name, user_id），
  - 事件对象的时间顺序记录，
  - 跟踪特定临时数据和状态，
  - 以及表示上次更新的时间戳。

- 开发人员通常通过 SessionService 间接与 Session 对象交互。
- SessionService 负责管理生命周期中的各种会话操作，如启动新会话、恢复会话、记录会话活动（包括用户活动）、标识活动会话以及管理会话数据的移除。
- ADK 提供了具有不同存储机制的 SessionService 实现，如记忆中存储和基于持久性的存储。
- 记忆中的 SessionService 适用于测试，但不支持应用重启后的数据持久化。

In [2]:
## 示例：使用 InMemorySessionService
## 这适用于不需要跨应用程序重启的数据持久性的本地开发和测试。
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()

In [ ]:
## 示例：使用 DatabaseSessionService
## 这适用于需要持久存储的生产或开发。
## 您需要配置数据库 URL（例如，用于 SQLite、PostgreSQL 等）。
## 需要：pip install google-adk[sqlalchemy] 和数据库驱动程序（例如，PostgreSQL 的 psycopg2）
from google.adk.sessions import DatabaseSessionService

## 示例使用本地 SQLite 文件：
db_url = "sqlite:///./my_agent_data.db"
session_service = DatabaseSessionService(db_url=db_url)

In [ ]:
## 示例：使用 VertexAiSessionService
## 这适用于 Google Cloud Platform 上的可扩展生产，利用
## Vertex AI 基础设施进行会话管理。
## 需要：pip install google-adk[vertexai] 和 GCP 设置/身份验证
from google.adk.sessions import VertexAiSessionService

PROJECT_ID = "your-gcp-project-id"  # 替换为您的 GCP 项目 ID
LOCATION = "us-central1"  # 替换为您想要的 GCP 位置

## 与此服务一起使用的 app_name 应对应于 Reasoning Engine ID 或名称
REASONING_ENGINE_APP_NAME = "projects/your-gcp-project-id/locations/us-central1/reasoningEngines/your-engine-id"  # 替换为您的 Reasoning Engine 资源名称

session_service = VertexAiSessionService(project=PROJECT_ID, location=LOCATION)

## 使用此服务时，将 REASONING_ENGINE_APP_NAME 传递给服务方法：
## session_service.create_session(app_name=REASONING_ENGINE_APP_NAME, ...)
## session_service.get_session(app_name=REASONING_ENGINE_APP_NAME, ...)
## session_service.append_event(session, event, app_name=REASONING_ENGINE_APP_NAME)
## session_service.delete_session(app_name=REASONING_ENGINE_APP_NAME, ...)

每次消息交换都涉及一个循环过程：接收消息，Runner 使用 SessionService 检索或建立 Session，Agent 使用 Session 的上下文（状态和历史交互）处理消息，Agent 生成响应并可能更新状态，Runner 将其封装为 Event，session_service.append_event 方法记录新事件并更新存储中的状态。然后 Session 等待下一条消息。理想情况下，当交互结束时使用 delete_session 方法终止会话。此过程说明了 SessionService 如何通过管理特定于 Session 的历史和临时数据来维护连续性。

- 在 ADK 中，每个 Session（会话）代表一个聊天线程，包含类似于代理临时工作记忆的状态组件，用于该特定对话的持续时间。
- Session.events 记录整个对话的历史，session.state 负责管理数据的更新并指向当前活动的状态。
- session.state 本质上是一个字典，存储键值对，支持启用代理存储和管理对话所需的关键信息，如用户偏好、任务进度、增量数据集合或影响后续操作的条件标志。
- 结构由键值对组成，值可以是可序列化的 Python 类型（数字、布尔值、列表、字典等），这些类型支持不同数据的持久化和变更。
- 状态组织可通过使用键前缀来定义数据作用域和持久性，未添加前缀的键默认为会话特定。
  - user: 前缀表示跨所有会话关联的用户 ID 数据。
  - app: 前缀表示在应用内共享的所有用户数据。
  - temp: 前缀表示仅对当前处理有效的数据，非持久化存储。
- 代理客户端通过单个 session.state 字典访问状态。SessionService 负责数据检索、合并和持久化，确保更新事件后数据保持最新。
- SessionService_agent.append_event() 确保准确跟踪保存，管理持久服务及状态变更处理。
- 简单方法：使用 output_key（针对代理文本回复）
  - 这是将代理最终文本回复直接插入状态的最简单方式。
  - 只需将 output_key 设为所需的 key，Runner 会自动执行更新操作，保存响应至状态中。

1. **简单方法**：使用 output_key（用于智能体回复）： 如果您只想将智能体的文本响应直接保存到状态中，这是最简单的方法。设置 LlmAgent 时，只需告诉它要使用的 output_key。Runner 看到这一点并在追加事件时自动创建必要的操作以将响应保存到状态。让我们看一个通过 output_key 演示状态更新的代码示例。

In [ ]:
## 从 Google 智能体开发工具包 (ADK) 导入必要的类
from google.adk.agents import LlmAgent
from google.adk.sessions import InMemorySessionService, Session
from google.adk.runners import Runner
from google.genai.types import Content, Part

## 定义带有 output_key 的 LlmAgent。
greeting_agent = LlmAgent(
    name="Greeter",
    model="gemini-2.0-flash",
    instruction="生成一个简短、友好的问候语。",
    output_key="last_greeting"
)

## --- 设置 Runner 和 Session ---
app_name, user_id, session_id = "state_app", "user1", "session1"
session_service = InMemorySessionService()
runner = Runner(
    agent=greeting_agent,
    app_name=app_name,
    session_service=session_service
)

session = session_service.create_session(
    app_name=app_name,
    user_id=user_id,
    session_id=session_id
)

print(f"初始状态：{session.state}")

## --- 运行智能体 ---
user_message = Content(parts=[Part(text="你好")])
print("\n--- 运行 agent ---")
for event in runner.run(
    user_id=user_id,
    session_id=session_id,
    new_message=user_message
):
    if event.is_final_response():
        print("Agent 已响应。")

## --- 检查更新的状态 ---
## 在 runner 完成处理所有事件*之后*正确检查状态。
updated_session = session_service.get_session(app_name, user_id, session_id)
print(f"\nAgent 运行后的状态：{updated_session.state}")

2. **标准方法**：使用 EventActions.state_delta（用于更复杂的更新）： 对于需要执行更复杂操作的时候——例如一次更新多个键、保存不只是文本的内容、针对特定范围（如 user: 或 app:）或进行与智能体文本回复无关的更新——您将手动构建状态更改的字典（state_delta）并将其包含在要追加的 Event 的 EventActions 中。让我们看一个例子：

In [ ]:
import time
from google.adk.tools.tool_context import ToolContext
from google.adk.sessions import InMemorySessionService

## --- 定义推荐的基于工具的方法 ---
def log_user_login(tool_context: ToolContext) -> dict:
    """
    在用户登录事件时更新会话状态。
    此工具封装了与用户登录相关的所有状态更改。
    参数：
        tool_context：由 ADK 自动提供，提供对会话状态的访问。
    返回：
        确认操作成功的字典。
    """
    # 通过提供的上下文直接访问状态。
    state = tool_context.state

    # 获取当前值或默认值，然后更新状态。
    # 这更加清晰并将逻辑共置。
    login_count = state.get("user:login_count", 0) + 1
    state["user:login_count"] = login_count
    state["task_status"] = "active"
    state["user:last_login_ts"] = time.time()
    state["temp:validation_needed"] = True

    print("从 `log_user_login` 工具内部更新了状态。")

    return {
        "status": "success",
        "message": f"已跟踪用户登录。总登录次数：{login_count}。"
    }

## --- 使用演示 ---
## 在真实应用程序中，LLM智能体将调用此工具。
## 在这里，我们模拟直接调用以进行演示。

## 1. 设置
session_service = InMemorySessionService()
app_name, user_id, session_id = "state_app_tool", "user3", "session3"
session = session_service.create_session(
    app_name=app_name,
    user_id=user_id,
    session_id=session_id,
    state={"user:login_count": 0, "task_status": "idle"}
)

print(f"初始状态：{session.state}")

## 2. 模拟工具调用（在真实应用中，ADK Runner 会执行此操作）
## 我们仅为此独立示例手动创建 ToolContext。
from google.adk.tools.tool_context import InvocationContext
mock_context = ToolContext(
    invocation_context=InvocationContext(
        app_name=app_name, user_id=user_id, session_id=session_id,
        session=session, session_service=session_service
    )
)

## 3. 执行工具
log_user_login(mock_context)

## 4. 检查更新的状态
updated_session = session_service.get_session(app_name, user_id, session_id)
print(f"工具执行后的状态：{updated_session.state}")

## 预期输出将显示与"之前"情况相同的状态更改，
## 但代码组织明显更清晰
## 且更健壮。

- 该代码演示了使用基于工具的方法管理应用程序中的用户会话状态。
- 定义了一个名为 `log_user_login` 的函数，该函数作为工具使用。该工具负责在用户登录时更新会话状态。
- 函数接收一个 `ToolContext` 对象，该对象允许访问和修改会话的状态字典。
- 使用该工具时，函数会递增 `user:login_count`，设置任务状态 `task_status` 为 `"active"`，记录用户的最新登录时间 `user:last_login_ts (timestamp)`，并添加一个临时字段 `temp:validation_needed`。
- 代码示例的演示部分展示了如何使用此工具。
- 它设置了一个记忆会话服务，并创建一个带有预定义状态的初始会话。
- 手动创建了一个 `ToolContext` 来模拟在 ADK 运行时执行工具的环境。
- 调用 `log_user_login` 函数模拟用户登录。
- 函数返回后，代码再次检索会话以显示状态已被更新，表明工具使状态更新更干净，且较之前直接修改状态的做法更加有序。
- 直接修改 `session.state` 字典不推荐，因为它绕过了标准事件处理机制，且不会在会话事件历史中记录。
- `SessionService` 选定时，状态不能被持久存储，建议使用推荐的方法通过工具的 `output_key` 参数或事件追加修改状态。
- 在设计状态时应保持简洁，使用基本数据类型，给键使用清晰名称和前缀，避免深度嵌套，并始终通过追加事件过程更新状态。

## 记忆：使用 MemoryService 的长期知识
- 在智能代理系统中，Session组件维护当前聊天历史（事件）和临时数据（状态），仅限于单次对话。
- 为了让代理能够跨多次交互保留信息或访问外部数据，需要长期知识管理，由MemoryService实现。
- 示例代码演示了本地开发和测试时使用InMemoryMemoryService，不需要应用重启即可保持数据。
- Session和State可以作为单次聊天的短期记忆。
- MemoryService管理长期知识，有持久化和可搜索的功能。
- MemoryService可以包含多种会话或外部来源的持久化信息。
- BaseMemoryService接口定义了管理可搜索长期知识的标准。
- 主要功能包括提取会话信息、存储信息、添加会话、检索信息和查询存储的数据。
- ADK提供了多种实现方案，InMemoryMemoryService适合暂存数据和本地测试。
- 生产环境推荐使用VertexAiRagMemoryService，它利用谷歌的云服务和RAG（检索增强生成）技术实现可扩展、持久和语义搜索的知识存储。

In [1]:
## 示例：使用 InMemoryMemoryService
## 这适用于不需要跨应用程序重启数据持久性的本地开发和测试
## 应用停止时内存内容会丢失
from google.adk.memory import InMemoryMemoryService

memory_service = InMemoryMemoryService()

c:\Users\yuw1si\.conda\envs\my_space\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
## 示例：使用 VertexAiRagMemoryService
## 这适用于 GCP 上的可扩展生产，利用
## Vertex AI RAG（检索增强生成）实现持久的、
## 可搜索的记忆。
## 需要：pip install google-adk[vertexai]，GCP
## 设置/身份验证和 Vertex AI RAG Corpus。
from google.adk.memory import VertexAiRagMemoryService

## 您的 Vertex AI RAG Corpus 的资源名称
RAG_CORPUS_RESOURCE_NAME = "projects/your-gcp-project-id/locations/us-central1/ragCorpora/your-corpus-id"  # 替换为您的 Corpus 资源名称

## 检索行为的可选配置
SIMILARITY_TOP_K = 5  # 要检索的顶部结果数
VECTOR_DISTANCE_THRESHOLD = 0.7  # 向量相似度阈值

memory_service = VertexAiRagMemoryService(
    rag_corpus=RAG_CORPUS_RESOURCE_NAME,
    similarity_top_k=SIMILARITY_TOP_K,
    vector_distance_threshold=VECTOR_DISTANCE_THRESHOLD
)

## 使用此服务时，像 add_session_to_memory
## 和 search_memory 这样的方法将与指定的 Vertex AI
## RAG Corpus 交互。

## 实操代码：LangChain 和 LangGraph 中的记忆管理
- 在 LangChain 和 LangGraph 中，内存是创建智能和自然对话应用的重要组成部分。
- 内存允许 AI 代理记住来自过去交互、学习反馈并适应偏好的信息。
- LangChain 的内存功能为存储历史对话提供基础，以丰富当前提示并记录测试交换。
- 内存对于完成复杂任务非常重要，提高效率和用户满意度。

- **短期内存**：
  - 具有线程范围，跟踪同一会话中的连续会话。
  - 提供即时上下文，但完整历史可能导致 LLM 的上下文窗口错误或性能降低。
  - LangGraph 管理短期内存，允许在状态持久化后续线程中恢复。

- **长期内存**：
  - 存储特定用户或应用的数据，跨会话共享。
  - 以自定义“命名空间”形式保存，可在任何线程中召回。
  - LangGraph 提供存储以保存和召回长期记忆，使代理能够无限期保留记忆。

- LangChain 提供了多种工具，支持从手动控制到链式自动化的会话历史管理。